# SISEPUEDE Tutorial #3 - Working with transformations

Welcome to the **SImulation of SEctoral Pathways with Uncertainty Exploration for DEcarbonization (SISEPUEDE)** tutorials! This tutorial walks users through the construction and analysis of transformations using the `Transformer`/`Transformers`, `Transformation`/`Transformations`, and `Strategy`/`Strategies` classes. By the end of this tutorial, users should be able to:

1. Access and use base `Transformer` and `Transformers` objects
2. Construct and evaluate transformations using the `Transformation` and `Transformations` classes
3. Build portfolios of policy transformations using the `Strategy` class
4. Construct and run sets of portfolios both within a notebook and using configuration (.yaml) files using the `Strategies` class

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
path = "/Users/dianamendez/sisepuede"
if path not in sys.path:
    sys.path.append(path)

import copy
import datetime as dt
import importlib # needed so that we can reload packages
import ipywidgets as wdg
import matplotlib.pyplot as plt
import os, os.path
import numpy as np
import pandas as pd
import pathlib
import sisepuede.core.support_classes as sc
import sisepuede.transformers as trf
import sisepuede.utilities._toolbox as sf
import time
from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
from typing import Union

# Two different things: *Transformer* vs. *Transformation*




# Transformations: what are they?

Transformations are well-defined in the SISEPUEDE ecosystem. A ``Transformation`` is a parameterization of a``Transformer``. Both are classes in SISEPUEDE, with ``Transformations`` and ``Transformers`` acting as collections of these classes, respectively. Each ``Transformer`` class is a function that modifies trajectories to reflect policy outcomes; documentation for each of the 60+ functions is available on readthedocs. 

``Transformation`` objects allow users to define parameterizations of ``Transformer`` objects using configuration files in a directory. This directory contains a configuration for:

1. All transformations
1. General configuraiton and information on the baseline
1. The strategy definition file, which combines transformations



See sisepuede.readthedocs.io for more information on transformations, transformers, and strategies.



# Let's explore the objects that underly all transformations: ``Transformers``

In [ ]:
help(trf.Transformers)

##  To get available `Transformer` classes, we need to start with a base dataset

- `Transformer` objects for SISEPUEDE are stored in the `Transformers` object
- `Transformers` requires an input data frame to transform; this data frame should be a set of raw SISEPUEDE inputs
    - Use `SISEPUEDEExamples()` to pull example data; examples are listed in `SISEPUEDEExamples.all_examples`

In [ ]:
examples = SISEPUEDEExamples()
df_input = examples("input_data_frame")
# note to james -- make this a fake country name (or example_country e.g.)


Transformer: "Kernel", basic function of the transformation f(x)
Transformation: parameterization of a transformer f(x; a, b...)
Strategy: collection of transformations (f_1(f_2(f_3...))

In [ ]:
"""fp_peru = pathlib.Path(
    "/Users/usuario/git/sisepuede_region_nbs/article_6_tanzania_sri_lanka_peru/data/sisepuede_raw_global_inputs_peru.csv"
)
df_input = pd.read_csv(fp_peru)
"""

##  We can build the `Transformers` class now

In [ ]:
import sisepuede.transformers.transformers as trfs
importlib.reload(trfs)
transformers = trfs.Transformers(
    {},
    df_input = df_input,  # examples("input_data_frame"),
)

# set some shortcuts
mat = transformers.model_attributes
time_periods = sc.TimePeriods(mat);
file_struct = transformers.file_struct


###  `Transformers` allows you to access all `Transformer` objects
- A `Transformer` is akin to a lever in the XLRM framework
- Each `Transformer` is associated with a variable (or set of variables) that represent a feasible, literature-based change in outcome due to at least one intervention(s)
- The list of available `Transformer` classes is available in `Transformers.all_transformers`
- Get a transformer using `Transformers.get_transformer`

In [ ]:
import sisepuede.manager.sisepuede_models as sm
models = sm.SISEPUEDEModels(
    mat,
    allow_electricity_run = True,
    fp_julia = file_struct.dir_jl,
    fp_nemomod_reference_files = file_struct.dir_ref_nemo,
)

###  A `Transformer` is callable
- When you get it, you can call it
- When called, it will generate a dataframe that is transformed under default parameters associated with each Transformer

In [ ]:
transformers.all_transformers

# Use the `get_transformer` method to get a transformer based on its code

In [ ]:
transformers.get_transformer("TFR:AGRC:DEC_CH4_RICE")

In [ ]:
tr_reduce_rice_methane = transformers.get_transformer("TFR:AGRC:DEC_CH4_RICE")
#tr_reduce_rice_methane.attribute_transformer_code
#tr_reduce_rice_methane.description

# you can call it 
df_input_reduce_rice = tr_reduce_rice_methane()
df_input_reduce_rice

In [ ]:
import sisepuede.models.afolu as mafl
model_afolu = mafl.AFOLU(mat, )

In [ ]:
tr_baseline = transformers.get_transformer("TFR:BASE")
df_baseline = tr_baseline()

df_emissions_untransformed = model_afolu(df_baseline, )
df_emissions_transformed = model_afolu(df_input_reduce_rice)

In [ ]:
df_emissions_untransformed[[x for x in df_emissions_untransformed.columns if "ch4" in x and "rice" in x]].plot()

In [ ]:
?tr_reduce_rice_methane

In [ ]:
df_input_reduce_rice = tr_reduce_rice_methane()#magnitude = 0.45)
df_emissions_transformed = model_afolu(df_input_reduce_rice)
df_emissions_transformed[[x for x in df_emissions_transformed.columns if "ch4" in x and "rice" in x]].plot()

In [ ]:
[x for x in dir(transformers.model_enercons) if "modvar_trns" in x]

modvar = transformers.model_enercons.modvar_trns_fuel_fraction_electricity
modvar = mat.get_variable(modvar)

In [ ]:
modvar.get_from_dataframe(df_input).tail()

In [ ]:
df_input[[x for x in df_input.columns if x.startswith("frac_trns_fuelmix_public_")]].tail()

###  A `Transfomer` is parameterized using keyword arguments
- No need to pass `df_input` unless you want to apply it to a data frame not used to instantiation the `Transformers` collection
- However, other arguments can be varied
- Full documentation of all `Transformer` functions is available at the [SISEPUEDE readthedocs](https://sisepuede.readthedocs.io/en/latest/transformers.html)
- Let's look at the doc string of `tr_medium_duty.function`, the base function in the transformer that we called (can also use `?tr_medium_duty`, but it will not show the signature)
    - `categories` can be varied to any TRNS category
    - `dict_allocation_fuels_target` is used to allocate the magnitude of the fuel shift across target fuels; 
        - `dict_allocation_fuels_target = {"fuel_electricity": 1.0}` means that 100% of the magnitude will be shifted away from `fuels_source`to electricity
    - `fuels_source` give fuels that are shifted away from. By default, this `Transformer` only shifts away from diesel and gas
    - `magnitude`: fraction of source fuel mix that is shifted to fuels specified in `dict_allocation_fuels_target`
    - `vec_implementation_ramp`: the implementation ramp vector. See discussion below for more information on how this can be specified

In [ ]:
?tr_medium_duty.function

# Now, let's examine `Transformation` classes

- A `Transformation` is a parameterization of a `Transformer`
- Let's look at fuel shifting medium-duty
- A `Transformation` can be defined using a dictionary or a configuration file. 
    - **NOTE**: A value of `None` can be passed in a YAML configuration using `null`


-----

#### For this example, we'll walk through a dictionary. 

The following keys are required in a dictionary/yaml file:

- `citations`: list of bibtex citations to call. These bibtex citations can be in the default SISEPUEDE library `sisepuede/docs/source/citations.bib` or provided in a Transformation definition directory in `citations.bib`
- `description`: optional description to provide. 
    - **NOTE**: Descriptions can be include citations as \\cite{CITEKEY}
- `identifiers` (**required**): 
    - `transformation_code` (**required**): specify a transformation code. These codes are used to define strategies. **NOTE: CANNOT CONTAIN "|" CHARACTER**
    - `transformation_name` (optional): optional name for the transformation, but it is recommended to provide one. This name is used in building automated reports and display tables.
- `parameters` (**required**): Parameters--or keyword arguments to the function--are passed as a dictionary associated with this key. 
    - Parameters included in here *must* be keyword arguments to the `Transformer` that will be parameterized
- `transformer` (**required**): The `Transformer` code that this transformation will parameterize. 
        

In [ ]:
# Default setup
dict_setup = {
    "citations": ["autho_123", "xbm"],
    "identifiers": {
        "transformation_code": "TX:TRNS:SHIFT_FUEL_MEDIUM_DUTY",
        "transformation_name": "Shift fuel for medium duty vehicles"
    },
    "description": "This came from this paper and that came from that paper",
    "parameters": {
        "categories": [
            "road_heavy_freight",
            "road_heavy_regional",
            "public"
        ],
        "dict_allocation_fuels_target": None,
        "fuels_source": [
            "fuel_diesel", 
            "fuel_gasoline"
        ],
        "magnitude": 0.5,
        "vec_implementation_ramp": {
            "alpha_logistic": 0,
            "n_tp_ramp": None,
            "tp_0_ramp": None,
            "window_logistic": [-8, 8]
        }
    },
    "transformer": "TFR:TRNS:SHIFT_FUEL_MEDIUM_DUTY"
}


# built the Transformation using dict_setup
transformation_1 = trf.Transformation(
    dict_setup,
    transformers,
)



# Default setup
dict_setup_2 = {
    "citations": ["autho_123", "xbm"],
    "identifiers": {
        "transformation_code": "TX:TRNS:SHIFT_FUEL_MEDIUM_DUTY_LOW",
        "transformation_name": "Shift fuel for medium duty vehicles at a lower intensity"
    },
    "description": "This came from this paper and that came from that paper",
    "parameters": {
        "categories": [
            "road_heavy_freight",
            "road_heavy_regional",
            "public"
        ],
        "dict_allocation_fuels_target": None,
        "fuels_source": [
            "fuel_diesel", 
            "fuel_gasoline"
        ],
        "magnitude": 0.3,
        "vec_implementation_ramp": {
            "alpha_logistic": 0,
            "n_tp_ramp": None,
            "tp_0_ramp": None,
            "window_logistic": [-8, 8]
        }
    },
    "transformer": "TFR:TRNS:SHIFT_FUEL_MEDIUM_DUTY"
}


# built the Transformation using dict_setup
transformation_1 = trf.Transformation(
    dict_setup,
    transformers,
)

transformation_2 = trf.Transformation(
    dict_setup_2,
    transformers,
)

# Look at the transformed variables for one of the categories

The transformation shifts fuel sources for three categories--`road_heavy_freight`, `road_heavy_regional`, and `public`--into electricity and hydrogen. Let's look at fuel fractions under two conditions--without the tranformation + with the transformation-for only the `public` transportation category.

In [ ]:
# First, let's build the fields
cats = ["public"]
fields = []

for fuel_name in ["Diesel", "Gasoline", "Electricity", "Hydrogen"]:
    fields += mat.build_variable_fields(
        f"Transportation Mode Fuel Fraction {fuel_name}",
        restrict_to_category_values = cats,
    )


# execute the transformation (can also use transformation_1.function())
df_input_transformation_1 = transformation_1()

# build a plot
fig, ax = plt.subplots(2, 1, figsize = (12, 10))

# base
ax[0].set_title("Base Inputs")
df_input[fields].plot(ax = ax[0])

# transformed
ax[1].set_title("Transformation 1 Inputs")
df_input_transformation_1[fields].plot(ax = ax[1])

In [ ]:
# First, let's build the fields
cats = ["public"]
fields = []

for fuel_name in ["Diesel", "Gasoline", "Electricity", "Hydrogen"]:
    fields += mat.build_variable_fields(
        f"Transportation Mode Fuel Fraction {fuel_name}",
        restrict_to_category_values = cats,
    )


# execute the transformation (can also use transformation_1.function())
df_input_transformation_2 = transformation_2()

# build a plot
fig, ax = plt.subplots(2, 1, figsize = (12, 10))

# base
ax[0].set_title("Base Inputs")
df_input[fields].plot(ax = ax[0])

# transformed
ax[1].set_title("Transformation 1 Inputs")
df_input_transformation_2[fields].plot(ax = ax[1])

##  There are 2 approaches available to change the timing or shape of the implementation ramp

The `implementation_ramp` is commonly specified using `vec_implementation_ramp` across SISEPUEDE. 

- The implementation ramp represents what fraction of the magnitude of a transformation is implemented across time. It takes values in [0, 1] and must have the same length as the original data
- If not specified, it defaults to start at the current year + 2 and end at the final time period
- **IMPORTANT NOTE**: When building Transformation directories, you can specify a default `vec_implementation_ramp` in the general config (under the `general` key); one for the baseline (under the `baseline` key in config_general); and, if desired, a unique one for any of the Transformations in the directory. In these configuration files--as with setup dictionaries--`vec_implementation` can be specified as a dictionary or a vector of values.

----


###  Approach 1: set the implementation ramp using a dictionary (preferred approach)
 

Let's demonstrate one approach to setting the implementation ramp. First, we can build one using parameters in a dictionary. The key parameters are:

- `alpha` (mix fraction): fraction of the ramp that is logistic. For all linear, set to 0.0; for completely logistic, set to 1.0
- `tp_0_ramp`: final time period == 0 in the ramp
- `window_logistic`: $(w_0, w_1)$ the window of the standard logit function [ $f(x) = (1 + e^{-x})^{-1}$ ] used to create the shape. In general, the min and max should be symmetric:
     - if $|w_0| < |w_1|$ and $w_0 < 0, w_1 > 0$, then the ramp will grow quickly and reach near-full implementation early; 
     - if $|w_0| > |w_1|$ and $w_0 < 0, w_1 > 0$, then the ramp will grow slowly and reach near-full implementation late; 
- `d` (not in interactive): value where logit == 0.5. For advanced use.

```
{
    "vec_implementation_ramp": {
        "alpha_logistic": alpha,
        "n_tp_ramp": len(time_periods.all_time_periods),
        "tp_0_ramp": tp_0_ramp,
        "window_logistic": (window_min, window_max)
    }
}
```

----

##  Now, we can interact with a widget to explore how these shape parameters change the implementation ramp


In [ ]:
def interactive_df_plot(
    df: pd.DataFrame,
    fields_plot: list,
) -> wdg.interactive:
    """
    Build an interactive ipywidget time seris plot for df. Will ignore fields specified
        in `fields_ignore`.
    """
    
    # build some widgets
    slider_alpha = wdg.FloatSlider(
        description = "$\\alpha$",
        min = 0.0, 
        max = 1.0, 
        step = 0.01,
        value = 0.0,
    )
    slider_tp_0 = wdg.IntSlider(
        description = "$t_0^{(ramp)}$",
        min = min(time_periods.all_time_periods), 
        max = max(time_periods.all_time_periods),
        value = (dt.datetime.now().year + 2 - time_periods.all_years[0]) # default
    )
    slider_window_max = wdg.FloatSlider(
        description = "$w_{max}$",
        max = 10.0,
        min = 0.0, 
        value = 8.0,
    )
    slider_window_min = wdg.FloatSlider(
        description = "$w_{min}$",
        max = 0.0,
        min = -10.0, 
        value = -8.0,
    )
        

    # placeholder for other actions
    df_plot = df

    # function to allow interaction
    def show_transformation(
        alpha: float,
        tp_0_ramp: int,
        window_max: float,
        window_min: float,
    ) -> 'plt.plot()':
        """
        Plot output fields from the model run on the df_model_data data frame
        """
        

        # update
        dict_setup_cur = copy.deepcopy(dict_setup)
        (
            dict_setup_cur
            .get("parameters")
            .update(
                {
                    "vec_implementation_ramp": {
                        "alpha_logistic": alpha,
                        "n_tp_ramp": len(time_periods.all_time_periods),
                        "tp_0_ramp": tp_0_ramp,
                        "window_logistic": (window_min, window_max)
                    }
                }
            )
        )


        # build a new transformation with the implementation ramp
        transformation_cur = trf.Transformation(
            dict_setup_cur,
            transformers,
        )

        # execute the transformation and plot (can also use transformation_1.function())
        df_input_transformation_cur = transformation_cur()
        
        # initialize plot
        fig, ax = plt.subplots(1, 1, figsize = (12, 6))
        ax.set_title("Transformation with modified vec_implementation_ramp")
        df_input_transformation_cur[fields_plot].plot(ax = ax, )
        
        plt.show()

        return None
    
    

    out = wdg.interactive(
        show_transformation,
        alpha = slider_alpha,
        tp_0_ramp = slider_tp_0,
        window_max = slider_window_max,
        window_min = slider_window_min,
    )

    return out


interactive_df_plot(
    df_input,
    fields,
)

###  We can read more about the parameters in the `_toolbox` function `ramp_vector`

In [ ]:
# check the characteristics of ramp vector
?sf.ramp_vector

###  Approach 2: directly set the implementation ramp

Another approach to setting the implementation ramp is to to build one from scratch. It must take values in [0, 1], start with 0, and end with 1. This approach can be useful for non-traditional curves, or for a symmetric benchmarks that must be met over time. 

The time periods in the example range from 0-35 (2015 to 2050); let's suppose we have a transformation that begins in 2030 (the last value == 0 in this case) with 80% implemenation by 2040 and 100% by 2050. 

``vec_implementation_ramp = np.concatenate([np.zeros(16), np.arange(1, 11)*0.08, np.arange(1, 11)*0.02 + 0.8])``

**NOTE**: the toolbox (`sisepuede.utilities._toolbox`) includes the `ramp_vector` function, which is extremely useful for building vectors from scratch.

In [ ]:
# set the vector--should be a numpy array
vec_implementation_ramp = np.concatenate(
    [
        np.zeros(16),
        np.arange(1, 11)*0.08,
        np.arange(1, 11)*0.02 + 0.8
    ]
)

# update
dict_setup_2 = copy.deepcopy(dict_setup)
dict_setup_2.get("parameters").update({"vec_implementation_ramp": vec_implementation_ramp})



# build a new transformation with the implementation ramp
transformation_2 = trf.Transformation(
    dict_setup_2,
    transformers,
)

# execute the transformation (can also use transformation_1.function())
df_input_transformation_2 = transformation_2()

fig, ax = plt.subplots(2, 1, figsize = (12, 10))

# base
ax[0].set_title("Base Data")
df_input[fields].plot(ax = ax[0])

# transformed
ax[1].set_title("Transformation 2 Data")
df_input_transformation_2[fields].plot(ax = ax[1])

# How do we define a `Transformations` object (which are required to build `Strategies`)?
- `Transformations` are collections of `Transformation` objects
- They are defined in collections of configuration files stored in a _Strategy Directory_
    - The _Strategy Directory_ contains information about transformations, strategic combinations, and even citations
- The easiest way to begin is to instantiate a default strategy directory;
    - It will export one `Transformation` YAML configuration file for each `Transformer` object and prepopulate with defaults. From there, you can modify parameters or duplicate as needed. 
    - Each Transformation **must** have a unique Transformation code to be counted
    - Note that there can be multiple `Transformation` files associated with a single `Transformer`
    - A default strategy definition file is also created that includes:
        - Base case (required)
        - All singleton strategies (1:1 mapping with `Transformation` objects that are defined)
        - Sectoral combinations (the instantation chooses one `Transformation` per `Transformer` to include)
        - All (using the same one `Transformation` per `Transformer`)

In [ ]:
trf.Transformation(
    config = {
        "identifiers": {
            "transformation_code": "TX:TRNS:SHIFT_FUEL_MEDIUM_DUTY_EX",
        },
        "parameters": {
            "vec_implementation_ramp": {
                "tp_0_ramp": 5,
            }
        },
        "transformer": "TFR:TRNS:SHIFT_FUEL_MEDIUM_DUTY"
    },
    transformers=transformers,
)


In [ ]:
# set an ouput path and instantiate
dir_transformations_out = "/Users/dianamendez/Desktop/transformations_default"
if True:#not os.path.exists(dir_transformations_out):
    trf.instantiate_default_strategy_directory(
        transformers,
        dir_transformations_out,
    )

In [ ]:
# then, you can load this back in after modifying (play around with it)
transformations = trf.Transformations(
    dir_transformations_out,
    df_input = df_input,
    transformers = transformers,
)
transformers = transformations.transformers
tab = transformations.attribute_transformation.table
tab

In [ ]:
# look at all codes that were accepts
transformations.all_transformation_codes

In [ ]:
# explore transformations
tr_dec_deforestation = transformations.get_transformation("TX:LNDU:DEC_DEFORESTATION")
df_dec_deforestation = tr_dec_deforestation()
df_dec_deforestation["lndu_reallocation_factor"] = 1
df_out_dec_deforestation = model_afolu(df_dec_deforestation)

tr_dec_deforestation_light = transformations.get_transformation("TX:LNDU:DEC_DEFORESTATION_LIGHT")
df_dec_deforestation_light = tr_dec_deforestation_light()
df_dec_deforestation_light["lndu_reallocation_factor"] = 1
df_out_dec_deforestation_light = model_afolu(df_dec_deforestation_light)

In [ ]:
modvar = mat.get_variable("Land Use Area")

fig, ax = plt.subplots(figsize = (14, 7))
modvar.get_from_dataframe(df_out_dec_deforestation).plot.area(ax = ax)


In [ ]:
modvar = mat.get_variable("Land Use Area")

fig, ax = plt.subplots(figsize = (14, 7))
modvar.get_from_dataframe(df_out_dec_deforestation_light).plot.area(ax = ax)


# Next, let's build some individual strategies
- A `Strategy` is a combination of `Transformations`; this approach allows for extensive variation in parameterization--including timing, magnitude, and categorization--of `Transformers`.
- A `Strategy` requires the following to be instatiated:
    - `strategy_id`
    - `transformation_specification`: If None or "", returns baseline strategy. Otherwise, can be a list of `Transformation` codes OR a pipe (i.e., "|") -delimited string
    - `Transformations` object used to combine `Transformation` functions
- Similar to a `Transformation`, a `Strategy` is callable

In [ ]:
# a baseline strategy can be instantiated without 
strat = trf.Strategy(
    0,
    None,
    transformations,
)

strat1 = trf.Strategy(
    1,
    ["TX:LSMM:INC_MANAGEMENT_CATTLE_PIGS", "TX:LSMM:INC_CAPTURE_BIOGAS"],
    transformations,
)

strat2 = trf.Strategy(
    2,
    "TX:TRNS:SHIFT_FUEL_MEDIUM_DUTY|ENTC:TARGET_RENEWABLE_ELEC",
    transformations
)


In [ ]:
df = strat2()
df[[x for x in df.columns if x.startswith("frac_trns_fuelmix_road_heavy_freight")]].tail()

##  Similar to a `Transformation` and `Transformer`, a `Strategy` is callable
- Note that it is pre-populated with the `strategy_id` (the field associated with `model_attributes.dim_strategy_id`)

In [ ]:
strat1()

##  Strategies can be instantied from a **Strategy Directory -- Skip Point 3**

A **Strategy Directory** is a self-contained definition set for strategies. The directory contains configuration files for the baseline and transformations, an optional citations database, and a strategy definition file. This directory also is the default output location for any Templates that are built using these files.

Below, we can call the `Strategies` object to build strategies. 
- Note the `prebuild` option; setting this to True will build all strategies and store them in a dataframe upon instantiation. 

In [ ]:
#HEREHEREHERE
t0 = time.time()
strategies = trf.Strategies(
    transformations,
    export_path = "transformations",
    prebuild = True,
)

t_elapse = sf.get_time_elapsed(t0)
print(f"Strategies defined at {strategies.transformations.dir_init} initialized in {t_elapse} seconds")


In [ ]:
import sisepuede.manager.sisepuede_models as sm
models = sm.SISEPUEDEModels(
    mat,
    allow_electricity_run = True,
    fp_julia = file_struct.dir_jl,
    fp_nemomod_reference_files = file_struct.dir_ref_nemo,
)


In [ ]:
#strat1 = strategies.get_strategy(6003)

df_output1 = models(strat1())

df_output2 = models(strat2())

##  Before reading into SISEPUEDE, input templates must built

- If you try to instantiate SISEPUEDE using a Strategy directory without templates, an error will occur
- use the `Strategies.build_strategies_to_templates()` method to build outputs
    - Note two important keyword arguments that are passed to `input_template.template_from_inputs()`:
        - df_trajgroup: optional dataframe mapping variable specifications to trajectory groups. The user only has to specify fields for which they want to defined a trajectory group in addition to the group (integer) that they want to assign the field to. (default None)
        - include_simplex_group_as_trajgroup: default to include simplex group from attributes as trajectory group? (default True)
        - see `?strategies.build_strategies_to_templates` for more details on these arguments

In [ ]:
?strategies.build_strategies_to_templates

##  Let's build our templates using `Strategies.build_strategies_to_templates()`
- First, let's specify some variable trajectory groups--these are groups of fields that share an LHC trial when varying in experiments
- We can pull a default example using the `SISEPUEDEExamples.variable_trajectory_group_specification` (see below)


In [ ]:
# call the example
df_vargroups = examples("variable_trajectory_group_specification")
df_vargroups


###   Let's see available strategies and pick some

- Additionally, the strategy attribute table from the default instantiation defines over 60 strategies; building all of these could be time-consuming. We can restrict the strategies we want to build by specifying IDs in the `strategies` keyword argument
- `strategies.attribute_table` is an attribute table; `strategies.attribute_table.table` is a data frame that can be accessed using standard Pandas methods

In [ ]:
strategies.attribute_table

In [ ]:
df_vargroups = examples("variable_trajectory_group_specification")

###  Finally, let's build the templates to `transformations.dir_init` directory
- you can specify elements of strategies using any combination of strategy id, name, or code
- The baseline is always included

In [ ]:
strategies.build_strategies_to_templates(
    #df_trajgroup = df_vargroups, 
    include_simplex_group_as_trajgroup = True,
    strategies = [0, "AF:ALL", "CE:ALL", "EN:ALL", "IP:ALL", "PFLO:ALL"],
)


In [ ]:
df = examples("input_data_frame")
df["energydensity_gravimetric_enfu_gj_per_tonne_fuel_natural_gas"] = 47.9

df.to_csv(
    pathlib.Path(examples.file_struct.dir_ref_examples).joinpath("input_data_frame.csv"),
    index = None,
    encoding = "UTF-8"
)

In [ ]:
import sisepuede as si
ssp = si.SISEPUEDE(
    "calibrated",
    attribute_time_period = att,
    #id_str = "sisepuede_run_2024-11-18T17:49:28.593255",
    initialize_as_dummy = True, # no connection to Julia is initialized if set to True
    regions = ["peru"],
    strategies = strategies,
    try_exogenous_xl_types_in_variable_specification = False,
)

In [ ]:
#?ssp.project_scenarios#GGTAB
df_out = ssp.read_output(None)

# Let's test some runs

In [ ]:
dict_run = {
    ssp.key_future: [0],
    ssp.key_design: [0],
    ssp.key_strategy: [
        0, 
        strategies.get_strategy_id("EN:ALL"), 
        strategies.get_strategy_id("PFLO:ALL")
    ],
}

# we'll save inputs since we're doing a small set of runs
ssp(
    dict_run,
    save_inputs = True,
)

In [ ]:
ssp.read_output(None)


In [ ]:
ssp.read_input(None)

In [ ]:
path_runs = "/Users/dianamendez/Downloads/sisepuede_summary_results_run_sisepuede_run_2025-07-21T13;50;17.070059"
path_runs = pathlib.Path(path_runs)

df_wide = pd.read_csv(path_runs.joinpath("WIDE_INPUTS_OUTPUTS.csv"))
df_primary = pd.read_csv(path_runs.joinpath("ATTRIBUTE_PRIMARY.csv"))
df_strategy = pd.read_csv(path_runs.joinpath("ATTRIBUTE_STRATEGY.csv"))


In [ ]:
strategies.attribute_table

In [ ]:
ssp.experimental_manager.dict_lhs_design.get("peru").arr_lhs_x

In [ ]:
df_strategy[
    df_strategy[ssp.key_strategy].isin(df_primary[ssp.key_strategy].unique())
]

In [ ]:
df_wide = pd.merge(
    df_wide,
    df_primary,
    how = "left"
)

In [ ]:
df_wide[ssp.key_strategy]
df_base_run = df_wide[df_wide[ssp.key_strategy].isin([0])]


In [ ]:
matt = ssp.model_attributes
fields_subsector_emission_total = matt.get_all_subsector_emission_total_fields()


In [ ]:
df_wide[df_wide[ssp.key_strategy].isin([0])][fields_subsector_emission_total]

import sisepuede.visualization.plots as spp

fig, ax = plt.subplots(figsize = (12, 7))
spp.plot_emissions_stack(
    df_base_run,
    matt,
    figtuple = (fig, ax, ),
)
ax.legend(loc = "upper right")

In [ ]:
modvar_lvst_pop = matt.get_variable("Livestock Head Count")
df_pop = modvar_lvst_pop.get_from_dataframe(
    df_wide[df_wide[ssp.key_strategy].isin([0])],
    fields_additional = [ssp.key_time_period]
)

df_pop.plot()


In [ ]:
df_base_run_new = df_base_run.copy()
df_base_run_new["frac_gnrl_eating_red_meat"] = 1
#df_base_run["frac_gnrl_eating_red_meat"] = 1

df_out_base = ssp.models.model_afolu(df_base_run_new.drop(columns = ["strategy_id", "primary_id", "future_id", "region"]))
modvar_elast = matt.get_variable("Elasticity of Livestock Demand to GDP per Capita")
modvar_lvst_demand = matt.get_variable("Livestock Demand")

# df_out_base
# modvar_elast.get_from_dataframe(df_base_run, ).plot()
modvar_lvst_demand.get_from_dataframe(df_out_base, ).plot.area()
